# EXEMPLO 1: Hybrid Query Básico — BGE-M3 (Ollama)

**Objetivo**: Referência rápida para implementar busca híbrida (BM25 + Vector) no OpenSearch usando **BGE-M3 via Ollama** (dim=1024) — mesmo padrão da Aula 1.

Este notebook demonstra:
- Setup mínimo de conexão OpenSearch + Ollama
- Criação de índice híbrido **com dimensão 1024** (BGE-M3)
- Geração de embeddings client-side via Ollama (`bge-m3`)
- **Pipeline RRF com sintaxe oficial OpenSearch 2.19+** (`score-ranker-processor`)
- Query híbrida completa
- Snippet copiável para reutilizar

**Referência (RRF):** <https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/>

> **Pré-requisitos**: Ollama rodando em `http://localhost:11434` com o modelo `bge-m3` carregado (`ollama pull bge-m3`).

## Setup Mínimo

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opensearch-py', 'requests'])

import os, requests
from opensearchpy import OpenSearch
from typing import List, Dict, Any

client = OpenSearch(
    hosts=[{'host': 'localhost', 'port': 9200}],
    http_auth=('admin', 'admin'),
    use_ssl=False,
    verify_certs=False
)

OLLAMA_BASE_URL    = os.getenv('OLLAMA_BASE_URL',  'http://localhost:11434')
OLLAMA_EMBED_MODEL = os.getenv('OLLAMA_EMBED_MODEL', 'bge-m3')
EMBED_DIM = 1024  # BGE-M3

print(f"OpenSearch: {client.info()['version']['number']}")
print(f"Ollama    : {OLLAMA_BASE_URL} ({OLLAMA_EMBED_MODEL}, dim={EMBED_DIM})")

OpenSearch: 3.0.0
Ollama    : http://localhost:11434 (bge-m3:latest, dim=1024)


## Função de Embedding (Ollama BGE-M3)

In [2]:
def ollama_embed(text: str, model: str = OLLAMA_EMBED_MODEL,
                 base_url: str = OLLAMA_BASE_URL) -> List[float]:
    """Gera embedding BGE-M3 via Ollama. Fallback: vetor zerado."""
    try:
        resp = requests.post(
            f"{base_url}/api/embeddings",
            json={"model": model, "prompt": text},
            timeout=30,
        )
        resp.raise_for_status()
        emb = resp.json().get("embedding", [])
        if len(emb) != EMBED_DIM:
            raise ValueError(f"Dim inesperada: {len(emb)} (esperado {EMBED_DIM})")
        return emb
    except Exception as e:
        print(f"⚠️  Ollama indisponível ({e}). Usando vetor zero.")
        return [0.0] * EMBED_DIM

vec = ollama_embed("teste de embedding")
print(f"✓ Dimensão do embedding: {len(vec)}")

✓ Dimensão do embedding: 1024


## Criar Índice Híbrido (dim=1024 para BGE-M3)

In [3]:
index_name = "indice_hibrido_exemplo1_bge_m3"

try:
    client.indices.delete(index=index_name)
except Exception:
    pass

mapping = {
    "settings": {"number_of_shards": 1, "index": {"knn": True}},
    "mappings": {
        "properties": {
            "texto": {"type": "text", "analyzer": "portuguese"},
            "embedding": {
                "type": "knn_vector",
                "dimension": EMBED_DIM,
                "method": {"name": "hnsw", "space_type": "cosinesimil", "engine": "faiss"}
            }
        }
    }
}

client.indices.create(index=index_name, body=mapping)
print(f"✓ Índice criado: {index_name} (dim={EMBED_DIM})")

✓ Índice criado: indice_hibrido_exemplo1_bge_m3 (dim=1024)


## Inserir Documentos

In [4]:
docs = [
    {"id": "1", "texto": "Lei de Acesso à Informação garante direito a informações públicas"},
    {"id": "2", "texto": "Direitos fundamentais incluem vida, liberdade, igualdade e segurança"},
    {"id": "3", "texto": "LGPD protege dados pessoais e privacidade dos cidadãos"},
]

for doc in docs:
    doc["embedding"] = ollama_embed(doc["texto"])
    client.index(index=index_name, id=doc["id"],
                 body={"texto": doc["texto"], "embedding": doc["embedding"]})

client.indices.refresh(index=index_name)
print(f"✓ {len(docs)} documentos indexados com embeddings BGE-M3")

✓ 3 documentos indexados com embeddings BGE-M3


## Criar Pipeline RRF (`score-ranker-processor`)

Sintaxe oficial OpenSearch 2.19+ — ver <https://opensearch.org/blog/introducing-reciprocal-rank-fusion-hybrid-search/>

In [5]:
try:
    client.transport.perform_request('DELETE', '/_search/pipeline/rrf_pipeline')
except Exception:
    pass

pipeline_body = {
    "description": "Post processor for hybrid RRF search (BGE-M3 + BM25)",
    "phase_results_processors": [
        {
            "score-ranker-processor": {
                "combination": {
                    "technique": "rrf",
                    "rank_constant": 60   # k da fórmula 1/(k + rank); default 60
                }
            }
        }
    ]
}

client.transport.perform_request('PUT', '/_search/pipeline/rrf_pipeline', body=pipeline_body)
print("✓ Pipeline RRF criado (score-ranker-processor, rank_constant=60)")

✓ Pipeline RRF criado (score-ranker-processor, rank_constant=60)


## Query Híbrida Completa

In [11]:
def hybrid_query(client, index: str, query_text: str, size: int = 5) -> List[Dict]:
    """Busca híbrida: BM25 (texto) + KNN (BGE-M3) + RRF (fusão)."""
    query_embedding = ollama_embed(query_text)

    search_body = {
        "size": size,
        "query": {
            "hybrid": {
                "queries": [
                    {"match": {"texto": {"query": query_text}}},
                    {"knn": {"embedding": {"vector": query_embedding, "k": size}}}
                ]
            }
        }
    }

    response = client.search(
        index=index,
        body=search_body,
        params={"search_pipeline": "rrf_pipeline"},
    )


    return [
        {'id': hit['_id'], 'score': hit['_score'], 'texto': hit['_source']['texto']}
        for hit in response['hits']['hits']
    ]

query = "informação pública e privacidade"
resultados = hybrid_query(client, index_name, query, size=3)



print(f"\nQuery: '{query}'")
print(f"\nResultados ({len(resultados)}):")
for r in resultados:
    print(f"\n[{r['id']}] Score: {r['score']:.4f}")
    print(f"    {r['texto']}")


Query: 'informação pública e privacidade'

Resultados (3):

[1] Score: 0.0328
    Lei de Acesso à Informação garante direito a informações públicas

[3] Score: 0.0323
    LGPD protege dados pessoais e privacidade dos cidadãos

[2] Score: 0.0159
    Direitos fundamentais incluem vida, liberdade, igualdade e segurança


In [8]:
query_embedding = ollama_embed("informação pública e privacidade")
print(query_embedding) 

[-1.4098948240280151, 0.6235729455947876, -0.9422231912612915, 0.24013012647628784, -0.48053237795829773, -0.20892733335494995, 0.7002131938934326, -0.1447412073612213, 1.0435302257537842, -0.8785594701766968, -0.14310164749622345, 0.18254613876342773, -0.18016725778579712, -0.7097437977790833, 0.11255016922950745, -0.0332084596157074, 0.7725374102592468, 0.3990786373615265, -0.18998730182647705, -0.14826202392578125, -0.6633862853050232, 0.2683302164077759, -0.5803918242454529, 0.20870424807071686, 0.7943916320800781, 0.0783054381608963, 1.4864771366119385, -0.15419316291809082, -0.43243587017059326, -1.07635498046875, -0.21484427154064178, -0.10571346431970596, 0.1402996927499771, -1.0562688112258911, -0.47235754132270813, -1.4974924325942993, -0.4746870994567871, -1.4131380319595337, -2.4250082969665527, 0.06231261044740677, 0.9270724654197693, 0.014190394431352615, 0.2675490379333496, -1.4258326292037964, 0.6174295544624329, 0.2887594401836395, -0.6535070538520813, -0.4407072663307

## Visualização dos Resultados

In [12]:
import pandas as pd

df = pd.DataFrame(resultados)[['id', 'score', 'texto']]
print("\nResultados em Tabela:")
print(df.to_string(index=False))
print(f"\nMelhor match score: {df['score'].max():.4f}")


Resultados em Tabela:
id    score                                                                texto
 1 0.032787    Lei de Acesso à Informação garante direito a informações públicas
 3 0.032258               LGPD protege dados pessoais e privacidade dos cidadãos
 2 0.015873 Direitos fundamentais incluem vida, liberdade, igualdade e segurança

Melhor match score: 0.0328


## Snippet Copiável para Reutilizar (BGE-M3 + RRF oficial)

In [ ]:
snippet = '''
import os, requests
from opensearchpy import OpenSearch

OLLAMA_BASE_URL    = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "bge-m3")
EMBED_DIM = 1024

def ollama_embed(text):
    r = requests.post(f"{OLLAMA_BASE_URL}/api/embeddings",
                      json={"model": OLLAMA_EMBED_MODEL, "prompt": text}, timeout=30)
    return r.json()["embedding"]

client = OpenSearch(
    hosts=[{"host": "localhost", "port": 9200}],
    http_auth=("admin", "admin"),
    use_ssl=False, verify_certs=False,
)

# Pipeline RRF oficial OpenSearch 2.19+
rrf = {
    "description": "Post processor for hybrid RRF search",
    "phase_results_processors": [{
        "score-ranker-processor": {
            "combination": {"technique": "rrf", "rank_constant": 60}
        }
    }]
}
client.transport.perform_request("PUT", "/_search/pipeline/rrf_pipeline", body=rrf)

def hybrid_search(query, index, size=5):
    qe = ollama_embed(query)
    body = {
        "size": size,
        "query": {
            "hybrid": {
                "queries": [
                    {"match": {"texto": {"query": query}}},
                    {"knn": {"embedding": {"vector": qe, "k": size}}},
                ]
            }
        }
    }
    return client.search(index=index, body=body,
                         params={"search_pipeline": "rrf_pipeline"})
'''

print(snippet)

## Resumo

**Componentes**:
- ✓ Embedding BGE-M3 via Ollama (dim=1024)
- ✓ Índice híbrido com BM25 + KNN (HNSW/FAISS)
- ✓ Pipeline RRF com **`score-ranker-processor`** (sintaxe oficial OpenSearch 2.19+)
- ✓ Query combinada via `hybrid` + `?search_pipeline=...`
- ✓ Função reutilizável

**Próximos passos**:
- Integrar Ollama ao OpenSearch como **connector ML Commons** (ver LAB1) para embeddings server-side
- Adicionar Neural Sparse Search com modelo `opensearch-neural-sparse-encoding-multilingual-v1` (ver LAB4)
- Implementar Contextual Retrieval com Groq + Ollama (ver LAB5)